In [ ]:
import sys
import subprocess
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "-q",
    "tensorflow-model-optimization",
    "tf-keras",
])

import os
import sys

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import glob
import re
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf

import tf_keras as keras

import tensorflow_model_optimization as tfmot

from sklearn.model_selection import GroupShuffleSplit

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

from sklearn.utils import shuffle

print("=" * 75)
print("TensorFlow :", tf.__version__)
print("tf_keras   :", keras.__version__)
print("TFMOT      :", tfmot.__version__)
print("=" * 75)

SISFALL_ROOT = (
    "/kaggle/input/datasets/kushajm/"
    "sisfall-dataset-fall-detection/"
    "SisFall_dataset"
)

KFALL_ROOT = (
    "/kaggle/input/datasets/usmanabbasi2002/"
    "kfall-dataset/KFall Dataset/"
    "KFall Dataset/sensor_data"
)

OUTPUT_DIR = "/kaggle/working"

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)

SEED = 42

np.random.seed(
    SEED
)

tf.random.set_seed(
    SEED
)

SISFALL_ORIG_FS = 200

KFALL_ORIG_FS = 100

TARGET_FS = 25

WINDOW_SECONDS = 6

WINDOW_LEN = (
    TARGET_FS
    * WINDOW_SECONDS
)

AM_THRESHOLD = 1.6

DOWNSAMPLE_MODE = "stride"

ACC_RANGE_G = 16.0

SISFALL_RIGHT_TRANSFORM = np.array(
    [
        [0, 0, -1],
        [0, 1, 0],
        [1, 0, 0],
    ],
    dtype=np.float32
)

KFALL_TRANSFORM = np.eye(
    3,
    dtype=np.float32
)

USE_AUGMENTATION = True

AUGMENT_COPIES = 1

MAX_ROTATION_DEG = 15.0

MAX_TIME_SHIFT = 8

JITTER_STD_G = 0.02

SCALE_MIN = 0.95

SCALE_MAX = 1.05

EPOCHS = 80

BATCH_SIZE = 32

LEARNING_RATE = 1e-3

DROPOUT_RATE = 0.35

QAT_EPOCHS = 5

QAT_LEARNING_RATE = 1e-5

MIN_VALIDATION_RECALL = 0.95

KFALL_FALL_TASK_MIN = 22

KFALL_FALL_TASK_MAX = 36

print()
print("=" * 75)
print("CONFIGURATION")
print("=" * 75)

print(
    "SisFall          :",
    SISFALL_ROOT
)

print(
    "KFall            :",
    KFALL_ROOT
)

print(
    "Target Fs        :",
    TARGET_FS,
    "Hz"
)

print(
    "Window           :",
    WINDOW_SECONDS,
    "sec"
)

print(
    "Window length    :",
    WINDOW_LEN
)

print(
    "Input features   :",
    "Ax Ay Az |A|"
)

print(
    "Input shape      :",
    f"{WINDOW_LEN} x 4"
)

print(
    "Downsampling     :",
    DOWNSAMPLE_MODE
)

print(
    "Orientation aug  :",
    f"±{MAX_ROTATION_DEG}°"
)

print(
    "Time shift       :",
    f"±{MAX_TIME_SHIFT} samples"
)

print("=" * 75)

if not os.path.isdir(
    SISFALL_ROOT
):

    raise FileNotFoundError(
        SISFALL_ROOT
    )

def get_subject_id(
    filepath
):

    parent = os.path.basename(
        os.path.dirname(
            filepath
        )
    )

    if re.match(
        r"^(SA|SE)\d+$",
        parent,
        re.IGNORECASE
    ):

        return parent.upper()

    filename = os.path.basename(
        filepath
    )

    match = re.search(
        r"(SA|SE)\d+",
        filename,
        re.IGNORECASE
    )

    if match:

        return (
            match
            .group(0)
            .upper()
        )

    return parent

def transform_xyz(
    xyz,
    matrix
):

    return (
        xyz
        @ matrix.T
    ).astype(
        np.float32
    )

def read_sisfall_raw_txt(
    filepath
):

    try:

        df = pd.read_csv(
            filepath,
            header=None,
            usecols=[
                0,
                1,
                2
            ],
            sep=",",
            engine="c"
        )

        accel_raw = df.to_numpy(
            dtype=np.float32
        )

        accel = (
            accel_raw
            * (
                32.0
                / 8192.0
            )
        )

        return accel.astype(
            np.float32
        )

    except Exception:

        try:

            rows = []

            with open(
                filepath,
                "r",
                encoding="utf-8",
                errors="ignore"
            ) as f:

                for line in f:

                    line = (
                        line
                        .strip()
                        .rstrip(",;")
                    )

                    if not line:

                        continue

                    parts = re.split(
                        r"[,;\s]+",
                        line
                    )

                    if len(
                        parts
                    ) < 3:

                        continue

                    try:

                        rows.append(
                            [
                                float(
                                    parts[
                                        0
                                    ]
                                ),
                                float(
                                    parts[
                                        1
                                    ]
                                ),
                                float(
                                    parts[
                                        2
                                    ]
                                ),
                            ]
                        )

                    except ValueError:

                        continue

            if len(
                rows
            ) == 0:

                return None

            accel_raw = np.asarray(
                rows,
                dtype=np.float32
            )

            return (
                accel_raw
                * (
                    32.0
                    / 8192.0
                )
            ).astype(
                np.float32
            )

        except Exception:

            return None

def acceleration_magnitude(
    accel
):

    return np.sqrt(
        np.sum(
            accel ** 2,
            axis=-1
        )
    )

def downsample_signal(
    data,
    original_fs,
    target_fs
):

    if original_fs == target_fs:

        return data.astype(
            np.float32
        )

    if (
        original_fs
        % target_fs
        != 0
    ):

        raise ValueError(
            "original_fs must be divisible by target_fs"
        )

    factor = (
        original_fs
        // target_fs
    )

    if (
        DOWNSAMPLE_MODE
        == "stride"
    ):

        return (
            data[
                ::factor
            ]
            .astype(
                np.float32
            )
        )

    usable = (
        len(
            data
        )
        // factor
        * factor
    )

    if usable == 0:

        return data.astype(
            np.float32
        )

    data = data[
        :usable
    ]

    result = (
        data
        .reshape(
            -1,
            factor,
            data.shape[
                1
            ]
        )
        .mean(
            axis=1
        )
    )

    return result.astype(
        np.float32
    )

def extract_window(
    accel
):

    magnitude = (
        acceleration_magnitude(
            accel
        )
    )

    candidates = np.where(
        magnitude
        > AM_THRESHOLD
    )[0]

    if len(
        candidates
    ) > 0:

        peak = int(
            candidates[
                np.argmax(
                    magnitude[
                        candidates
                    ]
                )
            ]
        )

    else:

        peak = int(
            np.argmax(
                magnitude
            )
        )

    half = (
        WINDOW_LEN
        // 2
    )

    start = (
        peak
        - half
    )

    end = (
        start
        + WINDOW_LEN
    )

    n = len(
        accel
    )

    if start < 0:

        start = 0

        end = WINDOW_LEN

    if end > n:

        end = n

        start = max(
            0,
            n
            - WINDOW_LEN
        )

    window = accel[
        start:end
    ]

    if len(
        window
    ) < WINDOW_LEN:

        if len(
            window
        ) == 0:

            return np.zeros(
                (
                    WINDOW_LEN,
                    3
                ),
                dtype=np.float32
            )

        missing = (
            WINDOW_LEN
            - len(
                window
            )
        )

        padding = np.repeat(
            window[
                -1:
            ],
            missing,
            axis=0
        )

        window = np.concatenate(
            [
                window,
                padding
            ],
            axis=0
        )

    return window.astype(
        np.float32
    )

def load_sisfall(
    root
):

    from tqdm.auto import tqdm

    txt_files = sorted(
        glob.glob(
            os.path.join(
                root,
                "**",
                "*.txt"
            ),
            recursive=True
        )
    )

    print()
    print("=" * 75)
    print("LOADING SISFALL RAW")
    print("=" * 75)

    print(
        "TXT files:",
        len(
            txt_files
        )
    )

    if len(
        txt_files
    ) == 0:

        raise FileNotFoundError(
            "No .txt files found."
        )

    X = []

    y = []

    groups = []

    skipped = 0

    first_file = True

    for fp in tqdm(
        txt_files,
        desc="Loading SisFall",
        unit="file"
    ):

        filename = os.path.basename(
            fp
        )

        if filename.upper().startswith(
            "F"
        ):

            label = 1

        elif filename.upper().startswith(
            "D"
        ):

            label = 0

        else:

            continue

        accel = read_sisfall_raw_txt(
            fp
        )

        if (
            accel is None
            or
            len(
                accel
            ) < 100
        ):

            skipped += 1

            continue

        accel = transform_xyz(
            accel,
            SISFALL_RIGHT_TRANSFORM
        )

        accel = downsample_signal(
            accel,
            SISFALL_ORIG_FS,
            TARGET_FS
        )

        window = extract_window(
            accel
        )

        X.append(
            window
        )

        y.append(
            label
        )

        groups.append(
            get_subject_id(
                fp
            )
        )

        if first_file:

            print()
            print(
                "FIRST FILE:"
            )

            print(
                fp
            )

            print(
                "Raw/downsampled length:",
                len(
                    accel
                )
            )

            print(
                "Window:",
                window.shape
            )

            print(
                "Median |a|:",
                float(
                    np.median(
                        acceleration_magnitude(
                            window
                        )
                    )
                ),
                "g"
            )

            first_file = False

    X = np.asarray(
        X,
        dtype=np.float32
    )

    y = np.asarray(
        y,
        dtype=np.int32
    )

    groups = np.asarray(
        groups
    )

    print()
    print("=" * 75)
    print("SISFALL SUMMARY")
    print("=" * 75)

    print(
        "Samples:",
        len(
            X
        )
    )

    print(
        "ADL:",
        int(
            np.sum(
                y == 0
            )
        )
    )

    print(
        "Fall:",
        int(
            np.sum(
                y == 1
            )
        )
    )

    print(
        "Subjects:",
        len(
            np.unique(
                groups
            )
        )
    )

    print(
        "Skipped:",
        skipped
    )

    print(
        "Shape:",
        X.shape
    )

    return (
        X,
        y,
        groups
    )

(
    X_raw,
    y_raw,
    groups_raw
) = load_sisfall(
    SISFALL_ROOT
)

cache_path = os.path.join(
    OUTPUT_DIR,
    "sisfall_25hz_raw_windows.npz"
)

np.savez_compressed(
    cache_path,
    X=X_raw,
    y=y_raw,
    groups=groups_raw
)

print(
    "Cache saved:",
    cache_path
)

def has_both_classes(
    y
):

    return (
        len(
            np.unique(
                y
            )
        )
        == 2
    )

def group_split(
    X,
    y,
    groups,
    test_size,
    seed
):

    for current_seed in range(
        seed,
        seed + 500
    ):

        splitter = GroupShuffleSplit(
            n_splits=1,
            test_size=test_size,
            random_state=current_seed
        )

        idx_a, idx_b = next(
            splitter.split(
                X,
                y,
                groups=groups
            )
        )

        if (
            has_both_classes(
                y[
                    idx_a
                ]
            )
            and
            has_both_classes(
                y[
                    idx_b
                ]
            )
        ):

            return (
                idx_a,
                idx_b
            )

    raise RuntimeError(
        "Could not find valid subject split."
    )

idx_temp, idx_test = group_split(
    X_raw,
    y_raw,
    groups_raw,
    test_size=0.30,
    seed=SEED
)

X_temp = X_raw[
    idx_temp
]

y_temp = y_raw[
    idx_temp
]

groups_temp = groups_raw[
    idx_temp
]

X_test_raw = X_raw[
    idx_test
]

y_test = y_raw[
    idx_test
]

groups_test = groups_raw[
    idx_test
]

val_ratio = (
    0.10
    / 0.70
)

idx_train, idx_val = group_split(
    X_temp,
    y_temp,
    groups_temp,
    test_size=val_ratio,
    seed=SEED + 1000
)

X_train_raw = X_temp[
    idx_train
]

y_train_raw = y_temp[
    idx_train
]

groups_train = groups_temp[
    idx_train
]

X_val_raw = X_temp[
    idx_val
]

y_val = y_temp[
    idx_val
]

groups_val = groups_temp[
    idx_val
]

train_subjects = set(
    groups_train
)

val_subjects = set(
    groups_val
)

test_subjects = set(
    groups_test
)

print()
print("=" * 75)
print("SUBJECT SPLIT")
print("=" * 75)

print(
    "Train subjects:",
    sorted(
        train_subjects
    )
)

print(
    "Validation subjects:",
    sorted(
        val_subjects
    )
)

print(
    "Test subjects:",
    sorted(
        test_subjects
    )
)

print(
    "Train ∩ Val :",
    train_subjects
    & val_subjects
)

print(
    "Train ∩ Test:",
    train_subjects
    & test_subjects
)

print(
    "Val ∩ Test:",
    val_subjects
    & test_subjects
)

assert not (
    train_subjects
    & val_subjects
)

assert not (
    train_subjects
    & test_subjects
)

assert not (
    val_subjects
    & test_subjects
)

print()

print(
    f"Train: {len(y_train_raw)} | "
    f"ADL={np.sum(y_train_raw == 0)} | "
    f"Fall={np.sum(y_train_raw == 1)}"
)

print(
    f"Val  : {len(y_val)} | "
    f"ADL={np.sum(y_val == 0)} | "
    f"Fall={np.sum(y_val == 1)}"
)

print(
    f"Test : {len(y_test)} | "
    f"ADL={np.sum(y_test == 0)} | "
    f"Fall={np.sum(y_test == 1)}"
)

def rotation_matrix_xyz(
    rx,
    ry,
    rz
):

    cx = np.cos(
        rx
    )

    sx = np.sin(
        rx
    )

    cy = np.cos(
        ry
    )

    sy = np.sin(
        ry
    )

    cz = np.cos(
        rz
    )

    sz = np.sin(
        rz
    )

    Rx = np.array(
        [
            [1, 0, 0],
            [0, cx, -sx],
            [0, sx, cx],
        ],
        dtype=np.float32
    )

    Ry = np.array(
        [
            [cy, 0, sy],
            [0, 1, 0],
            [-sy, 0, cy],
        ],
        dtype=np.float32
    )

    Rz = np.array(
        [
            [cz, -sz, 0],
            [sz, cz, 0],
            [0, 0, 1],
        ],
        dtype=np.float32
    )

    return (
        Rz
        @ Ry
        @ Rx
    ).astype(
        np.float32
    )

def random_rotation(
    sample
):

    rx = np.deg2rad(
        np.random.uniform(
            -MAX_ROTATION_DEG,
            MAX_ROTATION_DEG
        )
    )

    ry = np.deg2rad(
        np.random.uniform(
            -MAX_ROTATION_DEG,
            MAX_ROTATION_DEG
        )
    )

    rz = np.deg2rad(
        np.random.uniform(
            -MAX_ROTATION_DEG,
            MAX_ROTATION_DEG
        )
    )

    R = rotation_matrix_xyz(
        rx,
        ry,
        rz
    )

    return (
        sample
        @ R.T
    ).astype(
        np.float32
    )

def random_time_shift(
    sample
):

    shift = np.random.randint(
        -MAX_TIME_SHIFT,
        MAX_TIME_SHIFT + 1
    )

    if shift == 0:

        return sample.copy()

    output = np.roll(
        sample,
        shift,
        axis=0
    )

    if shift > 0:

        output[
            :shift
        ] = sample[
            0
        ]

    else:

        output[
            shift:
        ] = sample[
            -1
        ]

    return output.astype(
        np.float32
    )

def augment_raw_sample(
    sample
):

    x = sample.copy()

    x = random_rotation(
        x
    )

    x = random_time_shift(
        x
    )

    scale = np.random.uniform(
        SCALE_MIN,
        SCALE_MAX
    )

    x = (
        x
        * scale
    )

    noise = np.random.normal(
        loc=0.0,
        scale=JITTER_STD_G,
        size=x.shape
    ).astype(
        np.float32
    )

    x = (
        x
        + noise
    )

    return x.astype(
        np.float32
    )

def prepare_training_data(
    X,
    y
):

    X_parts = [
        X.copy()
    ]

    y_parts = [
        y.copy()
    ]

    if USE_AUGMENTATION:

        for copy_id in range(
            AUGMENT_COPIES
        ):

            print(
                f"Generating augmentation "
                f"{copy_id + 1}/{AUGMENT_COPIES}"
            )

            X_aug = np.empty_like(
                X,
                dtype=np.float32
            )

            for i in range(
                len(
                    X
                )
            ):

                X_aug[
                    i
                ] = augment_raw_sample(
                    X[
                        i
                    ]
                )

            X_parts.append(
                X_aug
            )

            y_parts.append(
                y.copy()
            )

    X_output = np.concatenate(
        X_parts,
        axis=0
    )

    y_output = np.concatenate(
        y_parts,
        axis=0
    )

    idx_adl = np.where(
        y_output == 0
    )[0]

    idx_fall = np.where(
        y_output == 1
    )[0]

    n_adl = len(
        idx_adl
    )

    n_fall = len(
        idx_fall
    )

    print()
    print("=" * 75)
    print("BEFORE CLASS BALANCING")
    print("=" * 75)

    print(
        "ADL:",
        n_adl
    )

    print(
        "Fall:",
        n_fall
    )

    if n_fall < n_adl:

        needed = (
            n_adl
            - n_fall
        )

        fall_pool = X_output[
            idx_fall
        ]

        additional_fall = []

        for _ in range(
            needed
        ):

            random_index = np.random.randint(
                len(
                    fall_pool
                )
            )

            additional_fall.append(
                augment_raw_sample(
                    fall_pool[
                        random_index
                    ]
                )
            )

        additional_fall = np.asarray(
            additional_fall,
            dtype=np.float32
        )

        X_output = np.concatenate(
            [
                X_output,
                additional_fall
            ],
            axis=0
        )

        y_output = np.concatenate(
            [
                y_output,
                np.ones(
                    needed,
                    dtype=np.int32
                )
            ]
        )

    X_output, y_output = shuffle(
        X_output,
        y_output,
        random_state=SEED
    )

    print()
    print("=" * 75)
    print("FINAL TRAIN DATA")
    print("=" * 75)

    print(
        "ADL:",
        np.sum(
            y_output == 0
        )
    )

    print(
        "Fall:",
        np.sum(
            y_output == 1
        )
    )

    print(
        "Total:",
        len(
            y_output
        )
    )

    return (
        X_output,
        y_output
    )

(
    X_train_aug_raw,
    y_train
) = prepare_training_data(
    X_train_raw,
    y_train_raw
)

def make_features(
    X_xyz
):

    X_xyz = np.asarray(
        X_xyz,
        dtype=np.float32
    )

    magnitude = np.sqrt(
        np.sum(
            X_xyz ** 2,
            axis=-1,
            keepdims=True
        )
    )

    xyz_norm = (
        np.clip(
            X_xyz,
            -ACC_RANGE_G,
            ACC_RANGE_G
        )
        / ACC_RANGE_G
    )

    mag_norm = (
        np.clip(
            magnitude,
            0.0,
            ACC_RANGE_G
        )
        / ACC_RANGE_G
    )

    features = np.concatenate(
        [
            xyz_norm,
            mag_norm
        ],
        axis=-1
    )

    return features.astype(
        np.float32
    )

X_train = make_features(
    X_train_aug_raw
)

X_val = make_features(
    X_val_raw
)

X_test = make_features(
    X_test_raw
)

print()
print("=" * 75)
print("FEATURE SHAPES")
print("=" * 75)

print(
    "Train:",
    X_train.shape
)

print(
    "Val:",
    X_val.shape
)

print(
    "Test:",
    X_test.shape
)

assert X_train.shape[
    1:
] == (
    WINDOW_LEN,
    4
)

def build_model():

    inp = keras.Input(
        shape=(
            WINDOW_LEN,
            4
        ),
        name="input"
    )

    x = keras.layers.Reshape(
        (
            WINDOW_LEN,
            1,
            4
        ),
        name="reshape"
    )(
        inp
    )

    x = keras.layers.Conv2D(
        filters=16,
        kernel_size=(
            7,
            1
        ),
        strides=(
            2,
            1
        ),
        padding="same",
        use_bias=False,
        name="conv1"
    )(
        x
    )

    x = keras.layers.BatchNormalization(
        name="bn1"
    )(
        x
    )

    x = keras.layers.ReLU(
        name="relu1"
    )(
        x
    )

    x = keras.layers.Conv2D(
        filters=32,
        kernel_size=(
            5,
            1
        ),
        strides=(
            2,
            1
        ),
        padding="same",
        use_bias=False,
        name="conv2"
    )(
        x
    )

    x = keras.layers.BatchNormalization(
        name="bn2"
    )(
        x
    )

    x = keras.layers.ReLU(
        name="relu2"
    )(
        x
    )

    x = keras.layers.Conv2D(
        filters=48,
        kernel_size=(
            3,
            1
        ),
        strides=(
            2,
            1
        ),
        padding="same",
        use_bias=False,
        name="conv3"
    )(
        x
    )

    x = keras.layers.BatchNormalization(
        name="bn3"
    )(
        x
    )

    x = keras.layers.ReLU(
        name="relu3"
    )(
        x
    )

    x = keras.layers.GlobalAveragePooling2D(
        name="gap"
    )(
        x
    )

    x = keras.layers.Dropout(
        DROPOUT_RATE,
        name="dropout"
    )(
        x
    )

    x = keras.layers.Dense(
        16,
        activation="relu",
        name="dense16"
    )(
        x
    )

    output = keras.layers.Dense(
        2,
        activation=None,
        name="logits"
    )(
        x
    )

    return keras.Model(
        inputs=inp,
        outputs=output,
        name="FallCNN_Light_AccMagnitude_Logits"
    )

base_model = build_model()

print()
print("=" * 75)
print("MODEL")
print("=" * 75)

base_model.summary()

print()
print(
    "base_model type:"
)

print(
    type(
        base_model
    )
)

print(
    "Is tf_keras Model:",
    isinstance(
        base_model,
        keras.Model
    )
)

if not isinstance(
    base_model,
    keras.Model
):

    raise RuntimeError(
        "The model is not a tf_keras model. "
        "Restart the Kaggle session and rerun from the top."
    )

total_params = (
    base_model.count_params()
)

base_model.compile(

    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),

    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),

    metrics=[
        "accuracy"
    ]
)

base_weights_path = os.path.join(
    OUTPUT_DIR,
    "best_float32.weights.h5"
)

base_callbacks = [

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=10,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=5,
        min_lr=1e-6,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=base_weights_path,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
]

print()
print("=" * 75)
print("FLOAT32 TRAINING")
print("=" * 75)

history = base_model.fit(

    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=EPOCHS,

    batch_size=BATCH_SIZE,

    callbacks=base_callbacks,

    verbose=1
)

if os.path.exists(
    base_weights_path
):

    base_model.load_weights(
        base_weights_path
    )

def evaluate_keras_model(model, model_name):

    print()
    print("=" * 75)
    print(f"{model_name} EVALUATION")
    print("=" * 75)

    train_loss, train_acc = model.evaluate(
        X_train,
        y_train,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    val_loss, val_acc = model.evaluate(
        X_val,
        y_val,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    test_loss, test_acc = model.evaluate(
        X_test,
        y_test,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    test_logits = model.predict(
        X_test,
        batch_size=BATCH_SIZE,
        verbose=0
    )

    test_prediction = np.argmax(
        test_logits,
        axis=1
    ).astype(np.int32)

    direct_test_acc = accuracy_score(
        y_test,
        test_prediction
    )

    print(f"Train loss     : {train_loss:.6f}")
    print(f"Train accuracy : {train_acc * 100:.2f}%")
    print(f"Val loss       : {val_loss:.6f}")
    print(f"Val accuracy   : {val_acc * 100:.2f}%")
    print(f"Test loss      : {test_loss:.6f}")
    print(f"Test accuracy  : {test_acc * 100:.2f}%")
    print(f"Argmax check   : {direct_test_acc * 100:.2f}%")

    return {
        "train_loss": float(train_loss),
        "train_accuracy": float(train_acc),
        "val_loss": float(val_loss),
        "val_accuracy": float(val_acc),
        "test_loss": float(test_loss),
        "test_accuracy": float(test_acc),
    }

float32_results = evaluate_keras_model(
    base_model,
    "FLOAT32 MODEL"
)

accuracy_plot = os.path.join(
    OUTPUT_DIR,
    "training_accuracy.png"
)

plt.figure(
    figsize=(
        9,
        5
    )
)

plt.plot(
    history.history[
        "accuracy"
    ],
    label="Train"
)

plt.plot(
    history.history[
        "val_accuracy"
    ],
    label="Validation"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Accuracy"
)

plt.title(
    "Float32 Training Accuracy"
)

plt.legend()

plt.grid()

plt.tight_layout()

plt.savefig(
    accuracy_plot,
    dpi=150
)

plt.show()

loss_plot = os.path.join(
    OUTPUT_DIR,
    "training_loss.png"
)

plt.figure(
    figsize=(
        9,
        5
    )
)

plt.plot(
    history.history[
        "loss"
    ],
    label="Train"
)

plt.plot(
    history.history[
        "val_loss"
    ],
    label="Validation"
)

plt.xlabel(
    "Epoch"
)

plt.ylabel(
    "Loss"
)

plt.title(
    "Float32 Training Loss"
)

plt.legend()

plt.grid()

plt.tight_layout()

plt.savefig(
    loss_plot,
    dpi=150
)

plt.show()

print()
print("=" * 75)
print("QAT COMPATIBILITY CHECK")
print("=" * 75)

print(
    "Model type:",
    type(
        base_model
    )
)

print(
    "Is legacy tf_keras model:",
    isinstance(
        base_model,
        keras.Model
    )
)

if not isinstance(
    base_model,
    keras.Model
):

    raise RuntimeError(
        "QAT cannot continue because base_model "
        "is not from tf_keras."
    )

print()
print("=" * 75)
print("CREATING QAT MODEL")
print("=" * 75)

quantize_model = (
    tfmot
    .quantization
    .keras
    .quantize_model
)

qat_model = quantize_model(
    base_model
)

qat_model.compile(

    optimizer=keras.optimizers.Adam(
        learning_rate=QAT_LEARNING_RATE
    ),

    loss=keras.losses.SparseCategoricalCrossentropy(
        from_logits=True
    ),

    metrics=[
        "accuracy"
    ]
)

qat_model.summary()

qat_weights_path = os.path.join(
    OUTPUT_DIR,
    "best_qat.weights.h5"
)

qat_callbacks = [

    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=2,
        restore_best_weights=True,
        verbose=1
    ),

    keras.callbacks.ModelCheckpoint(
        filepath=qat_weights_path,
        monitor="val_loss",
        save_best_only=True,
        save_weights_only=True,
        verbose=1
    ),
]

print()
print("=" * 75)
print("QAT TRAINING")
print("=" * 75)

qat_history = qat_model.fit(

    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=QAT_EPOCHS,

    batch_size=BATCH_SIZE,

    callbacks=qat_callbacks,

    verbose=1
)

if os.path.exists(
    qat_weights_path
):

    qat_model.load_weights(
        qat_weights_path
    )

qat_results = evaluate_keras_model(
    qat_model,
    "QAT MODEL"
)

print()
print("=" * 75)
print("FLOAT32 vs QAT ACCURACY")
print("=" * 75)
print(
    f"FLOAT32 Train : {float32_results['train_accuracy'] * 100:.2f}%"
)
print(
    f"QAT     Train : {qat_results['train_accuracy'] * 100:.2f}%"
)
print()
print(
    f"FLOAT32 Val   : {float32_results['val_accuracy'] * 100:.2f}%"
)
print(
    f"QAT     Val   : {qat_results['val_accuracy'] * 100:.2f}%"
)
print()
print(
    f"FLOAT32 Test  : {float32_results['test_accuracy'] * 100:.2f}%"
)
print(
    f"QAT     Test  : {qat_results['test_accuracy'] * 100:.2f}%"
)
print(
    f"Test acc diff : "
    f"{(qat_results['test_accuracy'] - float32_results['test_accuracy']) * 100:+.2f} percentage points"
)

def representative_dataset():

    rng = np.random.default_rng(
        SEED
    )

    count = min(
        500,
        len(
            X_train
        )
    )

    indices = rng.choice(
        len(
            X_train
        ),
        size=count,
        replace=False
    )

    for index in indices:

        sample = X_train[
            index:
            index + 1
        ].astype(
            np.float32
        )

        yield [
            sample
        ]

print()
print("=" * 75)
print("CONVERTING TO FULL INT8")
print("=" * 75)

converter = (
    tf.lite
    .TFLiteConverter
    .from_keras_model(
        qat_model
    )
)

converter.optimizations = [
    tf.lite.Optimize.DEFAULT
]

converter.representative_dataset = (
    representative_dataset
)

converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS_INT8
]

converter.inference_input_type = (
    tf.int8
)

converter.inference_output_type = (
    tf.int8
)

tflite_model = (
    converter.convert()
)

tflite_path = os.path.join(
    OUTPUT_DIR,
    "fall_acc_magnitude_logits_qat_int8.tflite"
)

with open(
    tflite_path,
    "wb"
) as f:

    f.write(
        tflite_model
    )

model_size_kb = (
    os.path.getsize(
        tflite_path
    )
    / 1024.0
)

print(
    "TFLite:",
    tflite_path
)

print(
    "Size:",
    f"{model_size_kb:.2f} KB"
)

interpreter = tf.lite.Interpreter(
    model_path=tflite_path
)

interpreter.allocate_tensors()

input_details = (
    interpreter
    .get_input_details()
)

output_details = (
    interpreter
    .get_output_details()
)

input_scale, input_zero = (
    input_details[
        0
    ][
        "quantization"
    ]
)

output_scale, output_zero = (
    output_details[
        0
    ][
        "quantization"
    ]
)

print()
print("=" * 75)
print("QUANTIZATION")
print("=" * 75)

print(
    "Input shape:",
    input_details[
        0
    ][
        "shape"
    ]
)

print(
    "Input dtype:",
    input_details[
        0
    ][
        "dtype"
    ]
)

print(
    "Input scale:",
    input_scale
)

print(
    "Input zero:",
    input_zero
)

print(
    "Output dtype:",
    output_details[
        0
    ][
        "dtype"
    ]
)

print(
    "Output scale:",
    output_scale
)

print(
    "Output zero:",
    output_zero
)

def tflite_logit_margins(
    X
):

    float_margins = np.zeros(
        len(
            X
        ),
        dtype=np.float32
    )

    int_margins = np.zeros(
        len(
            X
        ),
        dtype=np.int16
    )

    output_qparams = output_details[
        0
    ][
        "quantization_parameters"
    ]

    output_scales = output_qparams[
        "scales"
    ]

    output_zero_points = output_qparams[
        "zero_points"
    ]

    if len(
        output_scales
    ) != 1:

        raise RuntimeError(
            "Expected per-tensor quantization for model output."
        )

    if len(
        output_zero_points
    ) != 1:

        raise RuntimeError(
            "Expected one output zero-point."
        )

    for i in range(
        len(
            X
        )
    ):

        sample = X[
            i:
            i + 1
        ].astype(
            np.float32
        )

        q_input = np.round(
            sample
            / input_scale
            + input_zero
        )

        q_input = np.clip(
            q_input,
            -128,
            127
        ).astype(
            np.int8
        )

        interpreter.set_tensor(
            input_details[
                0
            ][
                "index"
            ],
            q_input
        )

        interpreter.invoke()

        q_output = (
            interpreter
            .get_tensor(
                output_details[
                    0
                ][
                    "index"
                ]
            )
        )

        q_adl = int(
            q_output[
                0,
                0
            ]
        )

        q_fall = int(
            q_output[
                0,
                1
            ]
        )

        q_margin = (
            q_fall
            - q_adl
        )

        int_margins[
            i
        ] = q_margin

        float_margins[
            i
        ] = (
            q_margin
            * output_scale
        )

    return (
        float_margins,
        int_margins
    )

(
    val_logit_margins,
    val_margin_int16
) = tflite_logit_margins(
    X_val
)

margin_min = int(
    np.min(
        val_margin_int16
    )
)

margin_max = int(
    np.max(
        val_margin_int16
    )
)

threshold_candidates_int16 = np.arange(
    margin_min - 1,
    margin_max + 2,
    dtype=np.int32
)

best_threshold_int16 = None

best_f1 = -1.0

best_recall = -1.0

for threshold_int16 in threshold_candidates_int16:

    prediction = (
        val_margin_int16
        >= threshold_int16
    ).astype(
        np.int32
    )

    recall = recall_score(
        y_val,
        prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_val,
        prediction,
        zero_division=0
    )

    if (
        recall
        >= MIN_VALIDATION_RECALL
        and
        f1
        > best_f1
    ):

        best_threshold_int16 = int(
            threshold_int16
        )

        best_f1 = float(
            f1
        )

        best_recall = float(
            recall
        )

if best_threshold_int16 is None:

    print(
        "WARNING: "
        "Validation recall >= 95% not achievable."
    )

    for threshold_int16 in threshold_candidates_int16:

        prediction = (
            val_margin_int16
            >= threshold_int16
        ).astype(
            np.int32
        )

        recall = recall_score(
            y_val,
            prediction,
            zero_division=0
        )

        f1 = f1_score(
            y_val,
            prediction,
            zero_division=0
        )

        if f1 > best_f1:

            best_threshold_int16 = int(
                threshold_int16
            )

            best_f1 = float(
                f1
            )

            best_recall = float(
                recall
            )

best_threshold = float(
    best_threshold_int16
    * output_scale
)

print()
print("=" * 75)
print("INT8 LOGIT-MARGIN THRESHOLD")
print("=" * 75)

print(
    "Threshold float logit margin:",
    best_threshold
)

print(
    "Threshold INT16 margin:",
    best_threshold_int16
)

print(
    "Validation Recall:",
    best_recall
)

print(
    "Validation F1:",
    best_f1
)

def calculate_metrics(
    y_true,
    scores,
    threshold
):

    prediction = (
        scores
        >= threshold
    ).astype(
        np.int32
    )

    accuracy = accuracy_score(
        y_true,
        prediction
    )

    precision = precision_score(
        y_true,
        prediction,
        zero_division=0
    )

    recall = recall_score(
        y_true,
        prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_true,
        prediction,
        zero_division=0
    )

    cm = confusion_matrix(
        y_true,
        prediction,
        labels=[
            0,
            1
        ]
    )

    tn, fp, fn, tp = (
        cm.ravel()
    )

    specificity = (
        tn
        / (
            tn
            + fp
        )
        if (
            tn
            + fp
        ) > 0
        else 0.0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "cm": cm,
        "prediction": prediction,
    }

(
    sisfall_logit_margins,
    sisfall_margin_int16
) = tflite_logit_margins(
    X_test
)

sisfall_metrics = calculate_metrics(
    y_test,
    sisfall_margin_int16,
    best_threshold_int16
)

print()
print("=" * 75)
print("SISFALL INT8 TEST")
print("=" * 75)

print(
    f"Accuracy    : "
    f"{sisfall_metrics['accuracy'] * 100:.2f}%"
)

print(
    f"Precision   : "
    f"{sisfall_metrics['precision'] * 100:.2f}%"
)

print(
    f"Recall      : "
    f"{sisfall_metrics['recall'] * 100:.2f}%"
)

print(
    f"Specificity : "
    f"{sisfall_metrics['specificity'] * 100:.2f}%"
)

print(
    f"F1          : "
    f"{sisfall_metrics['f1'] * 100:.2f}%"
)

print()

print(
    classification_report(
        y_test,
        sisfall_metrics[
            "prediction"
        ],
        target_names=[
            "ADL",
            "Fall"
        ],
        zero_division=0
    )
)

def plot_confusion_matrix(
    cm,
    title,
    path
):

    plt.figure(
        figsize=(
            6,
            5
        )
    )

    plt.imshow(
        cm,
    cmap="Blues"
    )

    plt.colorbar()

    for i in range(
        2
    ):

        for j in range(
            2
        ):

            plt.text(
                j,
                i,
                str(
                    cm[
                        i,
                        j
                    ]
                ),
                ha="center",
                va="center"
            )

    plt.xticks(
        [
            0,
            1
        ],
        [
            "ADL",
            "Fall"
        ]
    )

    plt.yticks(
        [
            0,
            1
        ],
        [
            "ADL",
            "Fall"
        ]
    )

    plt.xlabel(
        "Predicted"
    )

    plt.ylabel(
        "True"
    )

    plt.title(
        title
    )

    plt.tight_layout()

    plt.savefig(
        path,
        dpi=150
    )

    plt.show()

sisfall_cm_path = os.path.join(
    OUTPUT_DIR,
    "sisfall_int8_confusion.png"
)

plot_confusion_matrix(
    sisfall_metrics[
        "cm"
    ],
    "SisFall INT8",
    sisfall_cm_path
)

def parse_kfall_task(
    filepath
):

    filename = os.path.basename(
        filepath
    )

    match = re.search(
        r"T(\d+)",
        filename,
        re.IGNORECASE
    )

    if match:

        return int(
            match.group(
                1
            )
        )

    return None

def read_kfall_csv(
    filepath
):

    try:

        df = pd.read_csv(
            filepath
        )

    except Exception:

        return None

    normalized_columns = {
        re.sub(
            r"[^a-z0-9]",
            "",
            str(
                column
            ).lower()
        ): column
        for column in df.columns
    }

    needed = [
        "accx",
        "accy",
        "accz"
    ]

    if all(
        name
        in normalized_columns
        for name in needed
    ):

        columns = [
            normalized_columns[
                name
            ]
            for name in needed
        ]

        accel = (
            df[
                columns
            ]
            .apply(
                pd.to_numeric,
                errors="coerce"
            )
            .to_numpy(
                dtype=np.float32
            )
        )

    elif df.shape[
        1
    ] >= 5:

        accel = (
            df.iloc[
                :,
                2:5
            ]
            .apply(
                pd.to_numeric,
                errors="coerce"
            )
            .to_numpy(
                dtype=np.float32
            )
        )

    else:

        return None

    valid = np.all(
        np.isfinite(
            accel
        ),
        axis=1
    )

    return accel[
        valid
    ].astype(
        np.float32
    )

def load_kfall(
    root
):

    from tqdm.auto import tqdm

    files = sorted(
        glob.glob(
            os.path.join(
                root,
                "**",
                "*.csv"
            ),
            recursive=True
        )
    )

    print()
    print("=" * 75)
    print("LOADING KFALL")
    print("=" * 75)

    print(
        "CSV files:",
        len(
            files
        )
    )

    X = []

    y = []

    groups = []

    skipped = 0

    for fp in tqdm(
        files,
        desc="Loading KFall",
        unit="file"
    ):

        task = parse_kfall_task(
            fp
        )

        if task is None:

            skipped += 1

            continue

        label = int(
            KFALL_FALL_TASK_MIN
            <= task
            <= KFALL_FALL_TASK_MAX
        )

        accel = read_kfall_csv(
            fp
        )

        if (
            accel is None
            or
            len(
                accel
            ) < 100
        ):

            skipped += 1

            continue

        accel = transform_xyz(
            accel,
            KFALL_TRANSFORM
        )

        accel = downsample_signal(
            accel,
            KFALL_ORIG_FS,
            TARGET_FS
        )

        window = extract_window(
            accel
        )

        X.append(
            window
        )

        y.append(
            label
        )

        groups.append(
            os.path.basename(
                os.path.dirname(
                    fp
                )
            )
        )

    X = np.asarray(
        X,
        dtype=np.float32
    )

    y = np.asarray(
        y,
        dtype=np.int32
    )

    groups = np.asarray(
        groups
    )

    print()
    print("=" * 75)
    print("KFALL SUMMARY")
    print("=" * 75)

    print(
        "Samples:",
        len(
            X
        )
    )

    print(
        "ADL:",
        np.sum(
            y == 0
        )
    )

    print(
        "Fall:",
        np.sum(
            y == 1
        )
    )

    print(
        "Subjects:",
        len(
            np.unique(
                groups
            )
        )
    )

    print(
        "Skipped:",
        skipped
    )

    return (
        X,
        y,
        groups
    )

kfall_metrics = None

kfall_cm_path = None

if os.path.isdir(
    KFALL_ROOT
):

    (
        X_kfall_raw,
        y_kfall,
        groups_kfall
    ) = load_kfall(
        KFALL_ROOT
    )

    X_kfall = make_features(
        X_kfall_raw
    )

    (
        kfall_logit_margins,
        kfall_margin_int16
    ) = tflite_logit_margins(
        X_kfall
    )

    kfall_metrics = calculate_metrics(
        y_kfall,
        kfall_margin_int16,
        best_threshold_int16
    )

    print()
    print("=" * 75)
    print("CROSS-DATASET INT8")
    print("Train SisFall -> Test KFall")
    print("=" * 75)

    print(
        f"Accuracy    : "
        f"{kfall_metrics['accuracy'] * 100:.2f}%"
    )

    print(
        f"Precision   : "
        f"{kfall_metrics['precision'] * 100:.2f}%"
    )

    print(
        f"Recall      : "
        f"{kfall_metrics['recall'] * 100:.2f}%"
    )

    print(
        f"Specificity : "
        f"{kfall_metrics['specificity'] * 100:.2f}%"
    )

    print(
        f"F1          : "
        f"{kfall_metrics['f1'] * 100:.2f}%"
    )

    print()

    print(
        classification_report(
            y_kfall,
            kfall_metrics[
                "prediction"
            ],
            target_names=[
                "ADL",
                "Fall"
            ],
            zero_division=0
        )
    )

    kfall_cm_path = os.path.join(
        OUTPUT_DIR,
        "kfall_int8_confusion.png"
    )

    plot_confusion_matrix(
        kfall_metrics[
            "cm"
        ],
        "SisFall -> KFall INT8",
        kfall_cm_path
    )

else:

    print(
        "KFall directory not found."
    )

def tflite_to_header(
    model_bytes
):

    var_name = (
        "fall_acc_magnitude_logits_qat_int8"
    )

    lines = []

    lines.append(
        "#pragma once"
    )

    lines.append(
        "#include <stdint.h>"
    )

    lines.append(
        ""
    )

    lines.append(
        f"#define TARGET_FS ({TARGET_FS})"
    )

    lines.append(
        f"#define WINDOW_LEN ({WINDOW_LEN})"
    )

    lines.append(
        "#define INPUT_FEATURES (4)"
    )

    lines.append(
        f"#define ACC_RANGE_G ({ACC_RANGE_G:.6f}f)"
    )

    lines.append(
        ""
    )

    lines.append(
        "// Feature order:"
    )

    lines.append(
        "// Ax/16, Ay/16, Az/16, |A|/16"
    )

    lines.append(
        ""
    )

    lines.append(
        "// Right orientation:"
    )

    lines.append(
        "// Xnew = -Zold"
    )

    lines.append(
        "// Ynew =  Yold"
    )

    lines.append(
        "// Znew =  Xold"
    )

    lines.append(
        ""
    )

    lines.append(
        f"#define INPUT_SCALE "
        f"({input_scale:.10f}f)"
    )

    lines.append(
        f"#define INPUT_ZERO_POINT "
        f"({int(input_zero)})"
    )

    lines.append(
        f"#define OUTPUT_SCALE "
        f"({output_scale:.10f}f)"
    )

    lines.append(
        f"#define OUTPUT_ZERO_POINT "
        f"({int(output_zero)})"
    )

    lines.append(
        ""
    )

    lines.append(
        "#define ADL_CLASS_INDEX (0)"
    )

    lines.append(
        "#define FALL_CLASS_INDEX (1)"
    )

    lines.append(
        f"#define FALL_LOGIT_MARGIN_THRESHOLD_FLOAT "
        f"({best_threshold:.10f}f)"
    )

    lines.append(
        f"#define FALL_LOGIT_MARGIN_THRESHOLD_INT16 "
        f"({best_threshold_int16})"
    )

    lines.append(
        ""
    )

    lines.append(
        "// NO SOFTMAX: classify directly from quantized logits."
    )

    lines.append(
        "// int16_t margin = (int16_t)output[1] - (int16_t)output[0];"
    )

    lines.append(
        "// bool fall = margin >= FALL_LOGIT_MARGIN_THRESHOLD_INT16;"
    )

    lines.append(
        ""
    )

    lines.append(
        "#ifdef __cplusplus"
    )

    lines.append(
        f"alignas(8) const unsigned char "
        f"{var_name}[] = {{"
    )

    lines.append(
        "#else"
    )

    lines.append(
        f"const unsigned char "
        f"{var_name}[] "
        f"__attribute__((aligned(8))) = {{"
    )

    lines.append(
        "#endif"
    )

    values = [
        f"0x{b:02x}"
        for b in model_bytes
    ]

    for i in range(
        0,
        len(
            values
        ),
        16
    ):

        lines.append(
            "  "
            + ", ".join(
                values[
                    i:
                    i + 16
                ]
            )
            + ","
        )

    lines.append(
        "};"
    )

    lines.append(
        f"const unsigned int "
        f"{var_name}_len = "
        f"{len(model_bytes)};"
    )

    return "\n".join(
        lines
    )

header_path = os.path.join(
    OUTPUT_DIR,
    "fall_acc_magnitude_logits_qat_int8_model.h"
)

with open(
    header_path,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        tflite_to_header(
            tflite_model
        )
    )

rows = [

    {
        "Dataset": "SisFall-Float32",
        "Accuracy": float32_results["test_accuracy"],
        "Precision": np.nan,
        "Recall": np.nan,
        "Specificity": np.nan,
        "F1": np.nan,
    },

    {
        "Dataset": "SisFall-QAT",
        "Accuracy": qat_results["test_accuracy"],
        "Precision": np.nan,
        "Recall": np.nan,
        "Specificity": np.nan,
        "F1": np.nan,
    },

    {
        "Dataset": "SisFall-INT8",
        "Accuracy": sisfall_metrics[
            "accuracy"
        ],
        "Precision": sisfall_metrics[
            "precision"
        ],
        "Recall": sisfall_metrics[
            "recall"
        ],
        "Specificity": sisfall_metrics[
            "specificity"
        ],
        "F1": sisfall_metrics[
            "f1"
        ],
    }

]

if kfall_metrics is not None:

    rows.append(
        {
            "Dataset": "KFall",
            "Accuracy": kfall_metrics[
                "accuracy"
            ],
            "Precision": kfall_metrics[
                "precision"
            ],
            "Recall": kfall_metrics[
                "recall"
            ],
            "Specificity": kfall_metrics[
                "specificity"
            ],
            "F1": kfall_metrics[
                "f1"
            ],
        }
    )

metrics_path = os.path.join(
    OUTPUT_DIR,
    "metrics_summary.csv"
)

pd.DataFrame(
    rows
).to_csv(
    metrics_path,
    index=False
)

print()
print("=" * 75)
print("FINAL SUMMARY")
print("=" * 75)

print(
    "Sensor              : ADXL345 Accelerometer only"
)

print(
    "Gyroscope           : NO"
)

print(
    f"Sampling            : {TARGET_FS} Hz"
)

print(
    f"Window              : {WINDOW_SECONDS} sec"
)

print(
    f"Input               : {WINDOW_LEN} x 4"
)

print(
    "Features            : Ax, Ay, Az, |A|"
)

print(
    f"Orientation aug     : ±{MAX_ROTATION_DEG}°"
)

print(
    f"Time shift          : ±{MAX_TIME_SHIFT} samples"
)

print(
    "Architecture        : "
    "Conv16 -> Conv32 -> Conv48 "
    "-> GAP -> Dense16 -> Logits2 (NO Softmax)"
)

print(
    "Parameters          :",
    total_params
)

print(
    "TFLite INT8 size    :",
    f"{model_size_kb:.2f} KB"
)

print(
    "Logit margin thresh :",
    f"{best_threshold:.4f}"
)

print(
    "Threshold INT16     :",
    best_threshold_int16
)

print()
print(
    "--- SISFALL FLOAT32 / QAT ---"
)

print(
    f"Float32 Train Acc   : "
    f"{float32_results['train_accuracy'] * 100:.2f}%"
)

print(
    f"Float32 Val Acc     : "
    f"{float32_results['val_accuracy'] * 100:.2f}%"
)

print(
    f"Float32 Test Acc    : "
    f"{float32_results['test_accuracy'] * 100:.2f}%"
)

print(
    f"QAT Train Acc       : "
    f"{qat_results['train_accuracy'] * 100:.2f}%"
)

print(
    f"QAT Val Acc         : "
    f"{qat_results['val_accuracy'] * 100:.2f}%"
)

print(
    f"QAT Test Acc        : "
    f"{qat_results['test_accuracy'] * 100:.2f}%"
)

print()
print(
    "--- SISFALL INT8 ---"
)

print(
    f"Accuracy            : "
    f"{sisfall_metrics['accuracy'] * 100:.2f}%"
)

print(
    f"Precision           : "
    f"{sisfall_metrics['precision'] * 100:.2f}%"
)

print(
    f"Sensitivity/Recall  : "
    f"{sisfall_metrics['recall'] * 100:.2f}%"
)

print(
    f"Specificity         : "
    f"{sisfall_metrics['specificity'] * 100:.2f}%"
)

print(
    f"F1                  : "
    f"{sisfall_metrics['f1']:.4f}"
)

if kfall_metrics is not None:

    print()
    print(
        "--- SISFALL -> KFALL INT8 ---"
    )

    print(
        f"Accuracy            : "
        f"{kfall_metrics['accuracy'] * 100:.2f}%"
    )

    print(
        f"Precision           : "
        f"{kfall_metrics['precision'] * 100:.2f}%"
    )

    print(
        f"Sensitivity/Recall  : "
        f"{kfall_metrics['recall'] * 100:.2f}%"
    )

    print(
        f"Specificity         : "
        f"{kfall_metrics['specificity'] * 100:.2f}%"
    )

    print(
        f"F1                  : "
        f"{kfall_metrics['f1']:.4f}"
    )

print()
print(
    "TFLite:"
)

print(
    tflite_path
)

print()

print(
    "Header:"
)

print(
    header_path
)

print()

print(
    "Metrics:"
)

print(
    metrics_path
)

zip_path = os.path.join(
    OUTPUT_DIR,
    "fall_acc_magnitude_logits_qat_esp32.zip"
)

files_to_zip = [
    tflite_path,
    header_path,
    base_weights_path,
    qat_weights_path,
    accuracy_plot,
    loss_plot,
    sisfall_cm_path,
    metrics_path,
    cache_path,
]

if kfall_cm_path is not None:
    files_to_zip.append(kfall_cm_path)

with zipfile.ZipFile(
    zip_path,
    "w",
    compression=zipfile.ZIP_DEFLATED
) as zf:
    for path in files_to_zip:
        if path is not None and os.path.exists(path):
            zf.write(
                path,
                arcname=os.path.basename(path)
            )

print()
print("ZIP:")
print(zip_path)
print(
    "ZIP size:",
    f"{os.path.getsize(zip_path) / 1024:.2f} KB"
)
print("=" * 75)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

def get_int8_logit_margins_for_plot(X):

    scores = np.zeros(
        len(X),
        dtype=np.float32
    )

    for i in range(
        len(X)
    ):

        sample = X[
            i:i + 1
        ].astype(
            np.float32
        )

        q_input = np.round(
            sample / input_scale
            + input_zero
        )

        q_input = np.clip(
            q_input,
            -128,
            127
        ).astype(
            np.int8
        )

        interpreter.set_tensor(
            input_details[0]["index"],
            q_input
        )

        interpreter.invoke()

        q_output = interpreter.get_tensor(
            output_details[0]["index"]
        )

        q_output = q_output.astype(
            np.int16
        )

        q_adl = int(
            q_output[0, 0]
        )

        q_fall = int(
            q_output[0, 1]
        )

        q_margin = (
            q_fall
            - q_adl
        )

        scores[i] = (
            q_margin
            * output_scale
        )

    return scores

float_logits = base_model.predict(
    X_test,
    batch_size=BATCH_SIZE,
    verbose=0
)

float_pred = np.argmax(
    float_logits,
    axis=1
)

float_accuracy = accuracy_score(
    y_test,
    float_pred
)

float_precision = precision_score(
    y_test,
    float_pred,
    zero_division=0
)

float_recall = recall_score(
    y_test,
    float_pred,
    zero_division=0
)

float_f1 = f1_score(
    y_test,
    float_pred,
    zero_division=0
)

float_cm = confusion_matrix(
    y_test,
    float_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = (
    float_cm.ravel()
)

float_specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0.0
)

int8_scores = (
    get_int8_logit_margins_for_plot(
        X_test
    )
)

print(
    "INT8 scores type:",
    type(int8_scores)
)

print(
    "INT8 scores shape:",
    int8_scores.shape
)

print(
    "Best threshold:",
    best_threshold
)

int8_pred = (
    int8_scores
    >= float(
        best_threshold
    )
).astype(
    np.int32
)

int8_accuracy = accuracy_score(
    y_test,
    int8_pred
)

int8_precision = precision_score(
    y_test,
    int8_pred,
    zero_division=0
)

int8_recall = recall_score(
    y_test,
    int8_pred,
    zero_division=0
)

int8_f1 = f1_score(
    y_test,
    int8_pred,
    zero_division=0
)

int8_cm = confusion_matrix(
    y_test,
    int8_pred,
    labels=[0, 1]
)

tn, fp, fn, tp = (
    int8_cm.ravel()
)

int8_specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else 0.0
)

metrics_names = [
    "Accuracy",
    "Precision",
    "Recall",
    "Specificity",
    "F1-score"
]

float32_values = np.array(
    [
        float_accuracy,
        float_precision,
        float_recall,
        float_specificity,
        float_f1
    ],
    dtype=np.float32
) * 100

int8_values = np.array(
    [
        int8_accuracy,
        int8_precision,
        int8_recall,
        int8_specificity,
        int8_f1
    ],
    dtype=np.float32
) * 100

print()

print("=" * 75)

print(
    "FLOAT32 vs INT8"
)

print("=" * 75)

for i, metric in enumerate(
    metrics_names
):

    print(
        f"{metric:12s} | "
        f"Float32 = {float32_values[i]:6.2f}% | "
        f"INT8 = {int8_values[i]:6.2f}%"
    )

x = np.arange(
    len(metrics_names)
)

width = 0.35

plt.figure(
    figsize=(
        10,
        6
    )
)

bars_float = plt.bar(
    x - width / 2,
    float32_values,
    width,
    label="Float32"
)

bars_int8 = plt.bar(
    x + width / 2,
    int8_values,
    width,
    label="INT8"
)

for bar in bars_float:

    height = bar.get_height()

    plt.text(
        bar.get_x()
        + bar.get_width() / 2,
        height + 0.3,
        f"{height:.2f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )

for bar in bars_int8:

    height = bar.get_height()

    plt.text(
        bar.get_x()
        + bar.get_width() / 2,
        height + 0.3,
        f"{height:.2f}%",
        ha="center",
        va="bottom",
        fontsize=9
    )

plt.xticks(
    x,
    metrics_names
)

plt.ylabel(
    "Score (%)"
)

plt.xlabel(
    "Evaluation Metrics"
)

plt.title(
    "Float32 vs INT8 Performance Comparison"
)

plt.legend()

plt.grid(
    axis="y",
    alpha=0.3
)

plt.ylim(
    0,
    105
)

plt.tight_layout()

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

float32_size = 34.45
int8_size = 16.55

model_names = [
    "Float32",
    "INT8"
]

model_sizes = [
    float32_size,
    int8_size
]

reduction = (
    1
    -
    int8_size / float32_size
) * 100

compression_ratio = (
    float32_size
    / int8_size
)

print("=" * 60)
print("MODEL SIZE COMPARISON")
print("=" * 60)

print(
    f"Float32 : {float32_size:.2f} KB"
)

print(
    f"INT8    : {int8_size:.2f} KB"
)

print(
    f"Reduction : {reduction:.2f}%"
)

print(
    f"Compression ratio : {compression_ratio:.2f}x"
)

print("=" * 60)

x = np.arange(
    len(model_names)
)

plt.figure(
    figsize=(7, 6)
)

bars = plt.bar(
    x,
    model_sizes,
    width=0.55
)

for bar in bars:

    height = bar.get_height()

    plt.text(
        bar.get_x()
        + bar.get_width() / 2,

        height + 1,

        f"{height:.2f} KB",

        ha="center",
        va="bottom",

        fontsize=12
    )

plt.xticks(
    x,
    model_names,
    fontsize=12
)

plt.ylabel(
    "Model Size (KB)",
    fontsize=12
)

plt.xlabel(
    "Model Precision",
    fontsize=12
)

plt.title(
    "Model Size Comparison: Float32 vs INT8",
    fontsize=14
)

plt.grid(
    axis="y",
    alpha=0.3
)

plt.ylim(
    0,
    40
)

plt.tight_layout()

plt.show()